# Demand Forecasting Training

In [3]:
!pip -q install joblib scikit-learn pandas numpy

In [ ]:
import sklearn
print("scikit-learn version:", sklearn.__version__)
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
print("pandas version:", pd.__version__)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


DATA_FILE = Path("prepared_supply_chain.csv")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

assert DATA_FILE.exists(), "Upload prepared_supply_chain.csv to Colab first."
df = pd.read_csv(DATA_FILE)
df["Date"] = pd.to_datetime(df["Date"])
print(df.shape)
print(df["Date"].min(), "->", df["Date"].max())

scikit-learn version: 1.9.1
pandas version: 3.0.6
(91250, 27)
2024-01-01 00:00:00 -> 2024-12-30 00:00:00


In [11]:
GROUP_KEYS = ["SKU_ID", "Warehouse_ID"]
TARGET = "Units_Sold"
CAT = ["SKU_ID", "Warehouse_ID"]
NUM = [
    "Lag_1","Lag_7","Lag_14","Lag_30",
    "Rolling_Mean_7","Rolling_Mean_30","Rolling_Std_30",
    "Promotion_Flag",
    "Day_of_Week","Day_of_Month","Month","Week_of_Year","Quarter","Trend_Index"
]
FEATURES = CAT + NUM

def make_features(data):
    out = data.copy().sort_values(GROUP_KEYS + ["Date"]).reset_index(drop=True)
    g = out.groupby(GROUP_KEYS, observed=True)[TARGET]
    out["Lag_1"] = g.shift(1)
    out["Lag_7"] = g.shift(7)
    out["Lag_14"] = g.shift(14)
    out["Lag_30"] = g.shift(30)
    out["Rolling_Mean_7"] = g.transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
    out["Rolling_Mean_30"] = g.transform(lambda s: s.shift(1).rolling(30, min_periods=1).mean())
    out["Rolling_Std_30"] = g.transform(lambda s: s.shift(1).rolling(30, min_periods=2).std())
    out["Day_of_Week"] = out.Date.dt.dayofweek
    out["Day_of_Month"] = out.Date.dt.day
    out["Month"] = out.Date.dt.month
    out["Week_of_Year"] = out.Date.dt.isocalendar().week.astype(int)
    out["Quarter"] = out.Date.dt.quarter
    out["Trend_Index"] = (out.Date - out.Date.min()).dt.days
    return out

features = make_features(df)
model_df = features.dropna(subset=NUM).reset_index(drop=True)
print("Model-ready rows:", len(model_df))


Model-ready rows: 83750


In [12]:
# Chronological split: latest 20% of distinct dates is the held-out test period.
unique_dates = np.sort(model_df["Date"].unique())
split_idx = int(len(unique_dates) * 0.8)
split_date = pd.Timestamp(unique_dates[split_idx])
train = model_df[model_df.Date < split_date].copy()
test = model_df[model_df.Date >= split_date].copy()
print("Train:", train.Date.min(), "->", train.Date.max(), len(train))
print("Test :", test.Date.min(), "->", test.Date.max(), len(test))


Train: 2024-01-31 00:00:00 -> 2024-10-24 00:00:00 67000
Test : 2024-10-25 00:00:00 -> 2024-12-30 00:00:00 16750


In [13]:
def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ae = np.abs(y_pred - y_true)
    nz = y_true != 0
    return {
        "MAE": float(ae.mean()),
        "RMSE": float(np.sqrt(np.mean((y_pred-y_true)**2))),
        "MAPE": float(np.mean(ae[nz] / y_true[nz]) * 100) if nz.any() else np.nan,
        "MAPE_rows_used": int(nz.sum()),
        "MAPE_zero_rows_skipped": int((~nz).sum()),
    }

baseline_metrics = metrics(test[TARGET], test["Demand_Forecast"])
print("Existing Demand_Forecast baseline:", baseline_metrics)


Existing Demand_Forecast baseline: {'MAE': 2.372515223880597, 'RMSE': 2.9786685285518217, 'MAPE': 20.010263147827274, 'MAPE_rows_used': 16640, 'MAPE_zero_rows_skipped': 110}


In [14]:
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
    ("num", "passthrough", NUM),
])

candidates = {
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=150, max_depth=14, min_samples_leaf=5, random_state=42, n_jobs=-1
    ),
    "GradientBoostingRegressor": GradientBoostingRegressor(
        n_estimators=150, learning_rate=0.05, max_depth=3, min_samples_leaf=5, random_state=42, loss="huber"
    ),
}

results = {}
trained = {}
for name, estimator in candidates.items():
    pipe = Pipeline([("preprocess", pre), ("model", estimator)])
    pipe.fit(train[FEATURES], train[TARGET])
    pred = np.maximum(0, pipe.predict(test[FEATURES]))
    results[name] = metrics(test[TARGET], pred)
    trained[name] = pipe
    print(name, results[name])


RandomForestRegressor {'MAE': 5.893002665157983, 'RMSE': 7.327001659460733, 'MAPE': 44.797375125754215, 'MAPE_rows_used': 16640, 'MAPE_zero_rows_skipped': 110}
GradientBoostingRegressor {'MAE': 5.7536912821933, 'RMSE': 7.1597751900674735, 'MAPE': 43.694824232219354, 'MAPE_rows_used': 16640, 'MAPE_zero_rows_skipped': 110}


In [ ]:
# Selecting the best trainable ML candidate by held-out MAE, then RMSE.
selected_name = min(results, key=lambda n: (results[n]["MAE"], results[n]["RMSE"]))
selected_model = trained[selected_name]
print("Selected ML model:", selected_name)
print("ML metrics:", results[selected_name])
print("Baseline metrics:", baseline_metrics)


Selected ML model: GradientBoostingRegressor
ML metrics: {'MAE': 5.7536912821933, 'RMSE': 7.1597751900674735, 'MAPE': 43.694824232219354, 'MAPE_rows_used': 16640, 'MAPE_zero_rows_skipped': 110}
Baseline metrics: {'MAE': 2.372515223880597, 'RMSE': 2.9786685285518217, 'MAPE': 20.010263147827274, 'MAPE_rows_used': 16640, 'MAPE_zero_rows_skipped': 110}


In [16]:
# Refit the selected ML model on ALL model-ready historical rows before deployment.
selected_model.fit(model_df[FEATURES], model_df[TARGET])

model_path = MODEL_DIR / "demand_forecasting_model.joblib"
metadata_path = MODEL_DIR / "demand_forecasting_metadata.json"
joblib.dump(selected_model, model_path)

metadata = {
    "selected_model": selected_name,
    "target": TARGET,
    "features": FEATURES,
    "train_test_method": "chronological 80/20 by date",
    "split_date": str(split_date.date()),
    "test_start": str(test.Date.min().date()),
    "test_end": str(test.Date.max().date()),
    "baseline_metrics": baseline_metrics,
    "candidate_metrics": results,
    "training_rows_after_feature_warmup": int(len(model_df)),
    "promotion_assumption_for_future_forecast": 0,
    "scikit_learn_version": sklearn.__version__,
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved:", model_path)
print("Saved:", metadata_path)


Saved: models/demand_forecasting_model.joblib
Saved: models/demand_forecasting_metadata.json


In [19]:
pip install -q xgboost

## Hyperparameter Tuning for XGBoost

We will use `RandomizedSearchCV` to find optimal hyperparameters for the `XGBoostRegressor`. This technique efficiently explores a predefined range of parameters to identify the combination that yields the best performance, helping to improve the model's predictive accuracy and generalization.

In [23]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
from xgboost import XGBRegressor

# Define the parameter distribution for RandomizedSearchCV
# These ranges are common starting points for XGBoost tuning
param_distributions = {
    'model__n_estimators': randint(100, 300), # Number of boosting rounds
    'model__learning_rate': uniform(0.01, 0.2), # Step size shrinkage to prevent overfitting
    'model__max_depth': randint(3, 10), # Maximum depth of a tree
    'model__subsample': uniform(0.6, 0.4), # Subsample ratio of the training instance
    'model__colsample_bytree': uniform(0.6, 0.4), # Subsample ratio of columns when constructing each tree
    'model__gamma': uniform(0, 0.2), # Minimum loss reduction required to make a further partition
}

# Create an XGBoost regressor instance
xgb = XGBRegressor(random_state=42, n_jobs=-1)

# Create a pipeline for RandomizedSearchCV (including preprocessing)
# We tune parameters of the 'model' step within this pipeline.
pipe_for_tuning = Pipeline([
    ("preprocess", pre), # 'pre' is the ColumnTransformer defined earlier
    ("model", xgb)
])

# Perform RandomizedSearchCV
# n_iter: Number of parameter settings that are sampled. Trades off execution time with finding good parameters.
# cv: Number of folds for cross-validation.
# scoring: Metric to optimize. 'neg_mean_absolute_error' is appropriate for demand forecasting as we want to minimize MAE.
random_search = RandomizedSearchCV(
    estimator=pipe_for_tuning,
    param_distributions=param_distributions,
    n_iter=20, # Number of different parameter combinations to try (can be increased for more thorough search)
    cv=3, # 3-fold cross-validation
    scoring='neg_mean_absolute_error', # Optimize for MAE (negative because GridSearchCV maximizes the score)
    verbose=1, # Print progress messages
    random_state=42,
    n_jobs=-1 # Use all available cores for parallel processing
)

print("Starting RandomizedSearchCV for XGBoost...")
random_search.fit(train[FEATURES], train[TARGET])

print("Best parameters found:", random_search.best_params_)
print("Best MAE (from cross-validation, lower is better):", -random_search.best_score_)

# Extract the best XGBoost estimator (model only) from the best pipeline
best_xgb_model = random_search.best_estimator_.named_steps['model']
print("Best XGBoost estimator extracted and ready for re-evaluation.")

Starting RandomizedSearchCV for XGBoost...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters found: {'model__colsample_bytree': np.float64(0.9085081386743783), 'model__gamma': np.float64(0.014808930346818072), 'model__learning_rate': np.float64(0.08169314570885453), 'model__max_depth': 3, 'model__n_estimators': 127, 'model__subsample': np.float64(0.9452413703502374)}
Best MAE (from cross-validation, lower is better): 4.101600805918376
Best XGBoost estimator extracted and ready for re-evaluation.


In [24]:
from xgboost import XGBRegressor

# Add XGBoost to candidates with best parameters from RandomizedSearchCV
candidates["XGBoostRegressor"] = XGBRegressor(
    n_estimators=int(random_search.best_params_['model__n_estimators']),
    learning_rate=random_search.best_params_['model__learning_rate'],
    max_depth=int(random_search.best_params_['model__max_depth']),
    subsample=random_search.best_params_['model__subsample'],
    colsample_bytree=random_search.best_params_['model__colsample_bytree'],
    gamma=random_search.best_params_['model__gamma'],
    random_state=42,
    n_jobs=-1
)

# Re-evaluate all candidates including XGBoost
results = {}
trained = {}
for name, estimator in candidates.items():
    pipe = Pipeline([("preprocess", pre), ("model", estimator)])
    pipe.fit(train[FEATURES], train[TARGET])
    pred = np.maximum(0, pipe.predict(test[FEATURES]))
    results[name] = metrics(test[TARGET], pred)
    trained[name] = pipe
    print(name, results[name])

RandomForestRegressor {'MAE': 5.893002665157983, 'RMSE': 7.327001659460733, 'MAPE': 44.797375125754215, 'MAPE_rows_used': 16640, 'MAPE_zero_rows_skipped': 110}
GradientBoostingRegressor {'MAE': 5.7536912821933, 'RMSE': 7.1597751900674735, 'MAPE': 43.694824232219354, 'MAPE_rows_used': 16640, 'MAPE_zero_rows_skipped': 110}
XGBoostRegressor {'MAE': 5.739127874288986, 'RMSE': 7.145743843571682, 'MAPE': 43.83625740965769, 'MAPE_rows_used': 16640, 'MAPE_zero_rows_skipped': 110}


In [21]:
# Select the best trainable ML candidate by held-out MAE, then RMSE (including XGBoost).
selected_name = min(results, key=lambda n: (results[n]["MAE"], results[n]["RMSE"]))
selected_model = trained[selected_name]
print("Selected ML model:", selected_name)
print("ML metrics:", results[selected_name])
print("Baseline metrics:", baseline_metrics)

Selected ML model: XGBoostRegressor
ML metrics: {'MAE': 5.716602702126574, 'RMSE': 7.11308461136535, 'MAPE': 43.71022379660185, 'MAPE_rows_used': 16640, 'MAPE_zero_rows_skipped': 110}
Baseline metrics: {'MAE': 2.372515223880597, 'RMSE': 2.9786685285518217, 'MAPE': 20.010263147827274, 'MAPE_rows_used': 16640, 'MAPE_zero_rows_skipped': 110}


In [22]:
# Refit the selected ML model on ALL model-ready historical rows before deployment.
selected_model.fit(model_df[FEATURES], model_df[TARGET])

model_path = MODEL_DIR / "demand_forecasting_model.joblib"
metadata_path = MODEL_DIR / "demand_forecasting_metadata.json"
joblib.dump(selected_model, model_path)

metadata = {
    "selected_model": selected_name,
    "target": TARGET,
    "features": FEATURES,
    "train_test_method": "chronological 80/20 by date",
    "split_date": str(split_date.date()),
    "test_start": str(test.Date.min().date()),
    "test_end": str(test.Date.max().date()),
    "baseline_metrics": baseline_metrics,
    "candidate_metrics": results,
    "training_rows_after_feature_warmup": int(len(model_df)),
    "promotion_assumption_for_future_forecast": 0,
    "scikit_learn_version": sklearn.__version__,
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved:", model_path)
print("Saved:", metadata_path)

Saved: models/demand_forecasting_model.joblib
Saved: models/demand_forecasting_metadata.json
